# 🛠️ Data Cleaning Syntax Guide & Protocol

This notebook orchestrates the `DataCleaner` engine. The `CLEANING_CONFIGS` dictionary defines the deterministic pipeline for each dataset.

### 📜 The Configuration Ledger (Schema)

| Key | Type | Description | Available Options | Strict JSON Example |
|:---|:---|:---|:---|:---|
| `file` | `str` | Path to raw CSV. | Any valid local file path | `"Dataset/file.csv"` |
| `cleaning_mode` | `str` | State-preserving split cleaning mode (General vs ML). | `"general"`, `"ml"` | `"general"` |
| `target_columns` | `list` | ML target column(s) for split stratification (ML mode only). | List of string column names | `["target_col"]` |
| `standardize_headers` | `bool` | Rename headers to standardized `snake_case` early. | `True`, `False` | `False` |
| `columns_to_drop` | `list` | Columns to remove immediately. | List of string column names | `["id_redundant", "useless_col"]` |
| `duplicate_strategy` | `str` | Row-level deduplication strategy. | `"keep_first"`, `"keep_last"`, `"keep_all"` | `"keep_first"` |
| `column_renames` | `dict` | Custom user-defined column renaming. | Mapping of `{"original_name": "new_name"}` | `{"raw_col": "clean_col"}` |
| `foreign_keys` | `list` | Foreign Keys (Protected from statistical repair). | List of string column names | `["customer_id", "product_id"]` |
| `target_variables` | `list` | Labels (Protected from mutation/leakage). | List of string column names | `["churn_event"]` |
| `datetime_conversions` | `dict` | Datetime coercion rules. | Format: `"mixed"`, `"auto"`, or explicit string like `"%Y-%m-%d"` | `{"col": {"format": "auto"}}` |
| `text_cleaning` | `list` | Cols for NLP normalization (Unicode/Strip). | List of string column names | `["city", "description"]` |
| `value_mapping` | `dict` | Deterministic value replacements. | Mapping of `{"dirty_value": "clean_value"}` | `{"col": {"dirty_val": "clean_val"}}` |
| `imputation_rules` | `dict` | Statistical missing value repair. | Strategy: `"simple"` (methods: `"mean"`, `"median"`, `"mode"`), `"advanced"` (methods: `"knn"`, `"mice"`), `"constant"` (requires `"fill_value"`). | `{"col": {"strategy": "simple", "method": "mode"}}` |
| `coercion_rules` | `dict` | Force-coerce mixed-type columns. | Target: `"numeric"`, `"datetime"`. Optional: `"strip_currency": True` | `{"col": {"target": "numeric", "strip_currency": True}}` |
| `cardinality_rules` | `dict` | Entropy-based rare-label collapsing. | Strategy: `"rare_label"`. Requires `"min_freq"` (float) | `{"col": {"strategy": "rare_label", "min_freq": 0.01}}` |
| `transforms` | `dict` | Mathematical skew-reduction. | Method: `"log1p"`, `"box_cox"`, `"yeo_johnson"` | `{"col": {"method": "log1p"}}` |
| `outlier_treatment` | `dict` | Outlier clipping (IQR/MAD/Winsor). | Method: `"cap_iqr"`, `"cap_mad"` (requires `"multiplier"`) or `"winsorize"` (requires `"limits"`) | `{"col": {"method": "winsorize", "limits": [0.05, 0.05]}}` |
| `leakage_rules` | `dict` | Temporal pruning of look-ahead features. | Keys: `"drop_cols"` (list), `"prediction_time_col"` (str) | `{"drop_cols": ["f1"], "prediction_time_col": "time_ts"}` |
| `association_rules` | `dict` | Pruning of redundant associated classes. | Keys: `"threshold"` (float), `"priority"` (list) | `{"threshold": 0.8, "priority": ["c1", "c2"]}` |
| `multicollinearity_threshold` | `float` \| `str` \| `None` | Spearman \|ρ\| drop threshold. | `None`, `"auto"`, or float in `(0.0, 1.0]` | `None` |

### ⚙️ The Rigid Execution Pipeline
The engine enforces a strict sequential order to ensure state consistency:

0a. **Custom Renames**: Apply custom user-defined column renaming early.
0b. **Standardize Headers**: Early renaming of all remaining column names to clean snake_case and dynamic mapping of all downstream configs.
1.  **Drop Columns**: Early removal of junk.
2.  **Deduplicate**: Row uniqueness enforcement.
2b. **Leakage Handling**: Removal of look-ahead features.
3.  **Datetime Conversions**: Type coercion for dates.
4.  **Coercion Cleansing**: Numeric forcing (stripping symbols).
4b. **Value Mapping**: Manual token cleanup.
5.  **Text Cleaning**: Unicode normalization & NLP prep.
6.  **Imputation**: Statistical repair (Simple/KNN/MICE).
7.  **Cardinality Grouping**: Collapsing rare labels.
8.  **Transforms**: Skew reduction (Log/Box-Cox).
9.  **Outlier Treatment**: Clipping extreme values.
10. **Resolve Categorical Association**: Pruning redundant associated classes.
11. **Resolve Multicollinearity**: Pruning redundant numeric features.
12. **Memory Optimization**: Final downcasting (Automatic).

### 🚫 Strict AI Rules for Automated Generation
1.  **Absolute Target Shield**: Never apply ANY statistical mutation (imputation_rules, transforms, outlier_treatment, cardinality_rules) to columns listed in `target_variables`.
2.  **ID Protection**: Never apply continuous transforms (`log1p`, etc.) to `foreign_keys` identifiers.
3.  **Sequential State**: Assume `columns_to_drop` and `duplicate_strategy` occur *before* any statistical treatment.
4.  **FK Imputation**: If imputing a `foreign_keys`, `constant` strategy with `valid_fk_placeholders` is mandatory.
5.  **Temporal State Awareness**: Outlier clipping (Step 9) occurs AFTER mathematical transforms (Step 8). If a `transform` is applied to a column, ensure the corresponding `outlier_treatment` multiplier accounts for the compressed, post-transform distribution. The engine caches outlier thresholds on the training split during `fit()` and statefully applies them to test splits to prevent data leakage.
6.  **No Hallucinations**: If a column's distribution is unknown, leave the treatment commented out with a `# [WARNING]`.
7.  **Version & IP Protection**: Never assign `datetime_conversions`, `coercion_rules`, or `statistical continuous imputations` to semantic versioning columns or IP addresses. Always shield them by listing them under `foreign_keys` (to bypass outlier capping and skewness transforms) and use the 'constant' strategy under imputation_rules with safe string placeholders if missingness is encountered. The numeric coercion engine is equipped with regex checks to protect decimal-like version patterns and IP addresses from separator stripping.
8.  **Header Standardization Dynamic Remapping**: When `standardize_headers` is `True`, configurations (like `imputation_rules`, `transforms`, etc.) can be written using the raw, unstandardized column headers. The engine dynamically maps them to `snake_case`. However, the AI must ensure that no two original columns map to the same standardized name.
9.  **ML Model Target Splitting**: For datasets intended for downstream predictive models, set `cleaning_mode` to `"ml"` and populate target columns under `target_columns` (instead of `target_variables`). This automatically shields the target from mutations and triggers stratified/temporal split detection.


### 📋 Blank Configuration Template

Copy and paste this block into the `CLEANING_CONFIGS` dictionary below to add a new dataset to the pipeline.

```python
"dataset_name": {
    # ── Core Pipeline Settings ──────────────────────────────────────
    "file": "Dataset/your_file.csv",
    "cleaning_mode": "general",
    "target_columns": [],
    "standardize_headers": False,
    "columns_to_drop": [],
    "duplicate_strategy": "keep_first",
    "column_renames": {},

    # ── Key Protection & Target Shielding ───────────────────────────
    "foreign_keys": [],
    "target_variables": [],

    # ── Text, Value & Date Cleaning ─────────────────────────────────
    "datetime_conversions": {},
    "text_cleaning": [],
    "value_mapping": {},

    # ── Missing Value & Type Repair ─────────────────────────────────
    "imputation_rules": {},
    "coercion_rules": {},

    # ── Cardinality & Outlier Management ────────────────────────────
    "cardinality_rules": {},
    "transforms": {},
    "outlier_treatment": {},

    # ── Feature Pruning & Selection ──────────────────────────────────
    "leakage_rules": {},
    "association_rules": {},
    "multicollinearity_threshold": None 
}
```


In [3]:
# ══════════════════════════════════════════════════════════════════════
#  Cell 1: Configuration Zone (LLM & Human Editable)
# ══════════════════════════════════════════════════════════════════════
CLEANING_CONFIGS = {
    "supply_chain_cleaned": {
        # ── Core Pipeline Settings ──────────────────────────────────────
        "file": "Dataset/supply_chain_cleaned.csv",  # Direct ingest from Silver layer
        "cleaning_mode": "ml",  # ML mode: stratified train/test split on target
        "target_columns": ["late_delivery_risk"],  # Stratification anchor for split
        "standardize_headers": False,  # Headers already snake_case
        "columns_to_drop": [
            # Zero-variance audit lineage columns
            "ingested_at",
            "cleaned_at",
            # 86.2% missing
            "order_zipcode",
            # Direct target leakage columns
            "days_for_shipping_real",
            # Direct target leakage columns
            "delivery_status",
            "shipping_date",
            # Model high-cardinality & split-bias drops
            "order_status",
            "order_state",
            "customer_state",
            # Multicollinearity drops
            "order_profit_per_order",
            "product_price",
            "order_item_total",
            "sales_per_customer",
            "order_item_discount_rate",
            "order_region",
            # Free-text address / URLs
            "customer_street",
            "product_image"
        ],
        "duplicate_strategy": "keep_first",
        # Solve date-coercion regex bug by renaming early at Step 0a
        "column_renames": {
            "days_for_shipment_scheduled": "scheduled_shipping_duration",
            "days_for_shipping_real": "days_for_shipping_real"
        },

        # ── Key Protection & Target Shielding ───────────────────────────
        "foreign_keys": [
            "customer_id",
            "order_customer_id",
            "order_id",
            "order_item_id",
            "order_item_cardprod_id",
            "product_card_id",
            "category_id",
            "customer_zipcode",
            "department_id",
            "product_category_id",
            "latitude",
            "longitude"
        ],
        "target_variables": ["late_delivery_risk"],

        # ── Text, Value & Date Cleaning ─────────────────────────────────
        "datetime_conversions": {
            "order_date": {"format": "auto"}
        },
        "text_cleaning": [],
        "value_mapping": {},

        # ── Missing Value & Type Repair ─────────────────────────────────
        "imputation_rules": {},
        "coercion_rules": {},

        # ── Cardinality & Outlier Management ────────────────────────────
        "cardinality_rules": {
            "customer_city": {"strategy": "rare_label", "min_freq": 0.01},
            "order_city": {"strategy": "rare_label", "min_freq": 0.01},
            "order_country": {"strategy": "rare_label", "min_freq": 0.01},
            "order_state": {"strategy": "rare_label", "min_freq": 0.01},
            "customer_lname": {"strategy": "rare_label", "min_freq": 0.01},
            "customer_fname": {"strategy": "rare_label", "min_freq": 0.01},
            "product_name": {"strategy": "rare_label", "min_freq": 0.01}
        },
        "transforms": {},
        "outlier_treatment": {},

        # ── Feature Pruning & Selection ──────────────────────────────────
        "leakage_rules": {
            "prediction_time_col": "order_date",
            "drop_cols": []
        },
        "association_rules": {},
        "multicollinearity_threshold": None
    }
}


In [4]:
# ══════════════════════════════════════════════════════════════
#  Cell 2: Execution — Do not edit
# ══════════════════════════════════════════════════════════════

from data_cleaner_engine import CleaningOrchestrator

orchestrator = CleaningOrchestrator(CLEANING_CONFIGS)
orchestrator.execute_all()

In [5]:
# ══════════════════════════════════════════════════════════════
#  Cell 3: Post-Cleaning Feature Engineering
# ══════════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
import os

train_path = 'cleaned_dataset/cleaned_supply_chain_cleaned_train.csv'
test_path = 'cleaned_dataset/cleaned_supply_chain_cleaned_test.csv'
pre_cleaned_path = 'Dataset/supply_chain_cleaned.csv'

if os.path.exists(train_path) and os.path.exists(test_path):
    print("Loading train and test splits...")
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)
    
    # 1. Extract Temporal Features from order_date
    print("Extracting temporal features from order_date...")
    for df in [train_df, test_df]:
        dates = pd.to_datetime(df['order_date'])
        df['order_month'] = dates.dt.month
        df['order_day_of_week'] = dates.dt.dayofweek
        df['order_hour'] = dates.dt.hour

    # 2. Extract Financial Prioritization Ratios
    print("Creating financial ratios...")
    for df in [train_df, test_df]:
        df['discount_ratio'] = df['order_item_discount'] / (df['sales'] + 1)
        df['profit_margin'] = df['benefit_per_order'] / (df['sales'] + 1)

    # 3. High-Quality Target Encoding on uncollapsed location/product fields
    if os.path.exists(pre_cleaned_path):
        print("Mapping raw uncollapsed location and product names for target encoding...")
        raw_cols = ['order_item_id', 'order_city', 'customer_city', 'product_name']
        raw_df = pd.read_csv(pre_cleaned_path, usecols=raw_cols)
        
        # Drop the collapsed versions from splits
        train_df = train_df.drop(columns=['order_city', 'customer_city', 'product_name'], errors='ignore')
        test_df = test_df.drop(columns=['order_city', 'customer_city', 'product_name'], errors='ignore')
        
        # Merge back uncollapsed names
        train_df = train_df.merge(raw_df, on='order_item_id', how='left')
        test_df = test_df.merge(raw_df, on='order_item_id', how='left')
        
        # Perform Smoothed Target Encoding (smoothing parameter prevents overfitting)
        def target_encode(train, test, col, target='late_delivery_risk', smooth=20):
            global_mean = train[target].mean()
            stats = train.groupby(col)[target].agg(['mean', 'count'])
            stats['smooth_val'] = (stats['mean'] * stats['count'] + global_mean * smooth) / (stats['count'] + smooth)
            
            mapping = stats['smooth_val'].to_dict()
            train[f'{col}_encoded'] = train[col].map(mapping).fillna(global_mean)
            test[f'{col}_encoded'] = test[col].map(mapping).fillna(global_mean)
            return train, test

        target_cols = ['order_city', 'customer_city', 'product_name', 'category_name']
        for col in target_cols:
            train_df, test_df = target_encode(train_df, test_df, col, 'late_delivery_risk', smooth=20)
            
        # Drop the uncollapsed raw string/category features to prevent model high-cardinality issues
        # (and keep the newly created numeric encoded columns)
        train_df = train_df.drop(columns=['order_city', 'customer_city', 'product_name'], errors='ignore')
        test_df = test_df.drop(columns=['order_city', 'customer_city', 'product_name'], errors='ignore')
        
        print("Target encoding successfully generated!")
        
    # Save the engineered splits back to disk
    train_df.to_csv(train_path, index=False)
    test_df.to_csv(test_path, index=False)
    print("Engineered splits saved to cleaned_dataset/")
else:
    print("Cleaned splits not found! Please run Cell 2 first.")



Loading train and test splits...


C:\Users\fachr\AppData\Local\Temp\ipykernel_17660\1438985220.py:16: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  test_df = pd.read_csv(test_path)


Extracting temporal features from order_date...
Creating financial ratios...
Mapping raw uncollapsed location and product names for target encoding...
Target encoding successfully generated!
Engineered splits saved to cleaned_dataset/
